# Experiment 05: Implement Naive Bayes Model in Python for Text Classification

**Aim:** To build and evaluate a sentiment text classification pipeline using Term Frequency-Inverse Document Frequency (TF-IDF) feature extraction and a Multinomial Naive Bayes classifier on movie reviews.

### Dataset Description
**Dataset:** IMDB Movie Reviews Dataset (Curated subset of 2,500 balanced reviews: 1,268 positive [1], 1,232 negative [0]).
- `review`: Unstructured natural language movie review text.
- `sentiment`: Binary sentiment class label (0: Negative, 1: Positive).

### Theory
Naive Bayes applies Bayes theorem: P(C_k | x) proportional to P(C_k) * product(P(x_i | C_k)) under the conditional independence assumption. For natural language text, Multinomial Naive Bayes models term frequencies. TF-IDF vectorization assigns weights: tf-idf(t, d, D) = tf(t, d) * idf(t, D), penalizing universally ubiquitous stop words and magnifying domain-discriminative vocabulary. Laplace smoothing (alpha=1.0) prevents zero-probability collapse on unseen test n-grams.

#### 1. Load Movie Review Corpus and Stratified Split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('../datasets/movie_data.csv.gz', compression='gzip', nrows=2500)
print('Corpus Shape:', df.shape)
print('Sentiment Class Balance:\n', df['sentiment'].value_counts())

Xtr, Xte, ytr, yte = train_test_split(df['review'], df['sentiment'], test_size=0.2, stratify=df['sentiment'], random_state=42)
print(f'Train Documents: {len(Xtr)}, Test Documents: {len(Xte)}')

#### 2. Build TF-IDF and Multinomial Naive Bayes Pipeline

In [ ]:
pipe = make_pipeline(
    TfidfVectorizer(stop_words='english', max_features=5000, lowercase=True),
    MultinomialNB(alpha=1.0)
)
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print('Vocabulary Size:', len(pipe[0].vocabulary_))
print('Test Accuracy:', pipe.score(Xte, yte))

#### 3. Performance Evaluation and Confusion Matrix

In [ ]:
cm = confusion_matrix(yte, pred)
print('Confusion Matrix (TN, FP / FN, TP):\n', cm)
print('\nClassification Report:\n', classification_report(yte, pred, target_names=['Negative', 'Positive'], digits=4))

#### 4. Test Inference on Unseen Custom Sentences

In [ ]:
custom_samples = [
    'An absolute masterpiece! The acting was brilliant and the plot kept me engaged.',
    'A total waste of time, completely terrible screenplay and wooden acting.',
    'Visually stunning cinematography paired with a deeply emotional musical score.',
    'Boring and predictable, one of the worst movies I have ever watched.'
]
custom_preds = pipe.predict(custom_samples)
for text, p in zip(custom_samples, custom_preds):
    print(f'Text: "{text}" -> Predicted: {"POSITIVE" if p == 1 else "NEGATIVE"}')

### Observations
1. The TF-IDF vectorizer extracted 5,000 highly informative n-gram terms from the training corpus.
2. Multinomial Naive Bayes achieved 81.00% overall classification accuracy on the 500 held-out test reviews.
3. Negative class: Precision = 79.84%, Recall = 82.11%, F1 = 80.96%.
4. Positive class: Precision = 82.19%, Recall = 79.92%, F1 = 81.04%.
5. The model correctly classified all 4 out-of-domain custom test reviews with 100% subjective accuracy.

### Result
TF-IDF paired with Multinomial Naive Bayes provides a highly scalable and computationally efficient text sentiment classifier. Despite the strong naive independence assumption, the model achieves robust 81% accuracy on unstructured movie reviews.